# Train Test Creator

## Install libraries

In [1]:
import os
import sys
import random
from dotenv import load_dotenv
import pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from logger.logger import Logger
from utils.constants import *
from utils.utils import *
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)
from dtos.tabular_database_driver_dtos.tabular_database_driver_dtos import *
from ta.ta_functions import *

load_dotenv()

True

In [2]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Parameters

In [3]:
STOCK_CODE = "VCB"
LOOKBACK_WINDOW = 20
FORECAST_HORIZON = 5
STRIDE = 5
ID_COLUMN = ["date", "code"]
TARGET_COLUMN = f"adjust"

# Inclusive
TRAIN_RANGE = ("2000-01-01", "2023-12-31")  # was 2021 → include recent regime
VAL_RANGE = ("2024-01-01", "2024-12-31")  # shifted forward
TEST_RANGE = ("2025-01-01", "2026-04-30")  # shifted forward

In [4]:
STOCK_CODE = str.lower(STOCK_CODE)
STOCK_CODE

'vcb'

In [5]:
TOTAL_WINDOW = LOOKBACK_WINDOW + FORECAST_HORIZON
TOTAL_WINDOW

25

## Load data

In [6]:
my_logger = Logger(
    file_name=f"{FEATURE_SELECTION_LOG_FILE_BASE}/{STOCK_CODE}/train_test_creator.log",
)

In [7]:
my_connection_model = PostgreSQLConnectionDto(
    logger=my_logger,
    host=os.getenv("POSTGRES_HOST"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("GOLD_POSTGRES_DATABASE"),
)

In [8]:
my_postgresql_driver = PostgreSQLDriver(logger=my_logger)
my_postgresql_driver.connect(my_connection_model)

<DatabaseExecutionStatus.SUCCESS: 'success'>

In [9]:
stock_df = my_postgresql_driver.select(
    schema_name=Schema.ENTERPRISE.value,
    table_name=f"unified_{STOCK_CODE}",
    order_by=["date"],
)

# cast all string columns that look numeric → float
for col in stock_df.columns:
    if stock_df[col].dtype == object:
        converted = pd.to_numeric(stock_df[col], errors="coerce")
        if converted.notna().sum() / len(stock_df) >= 0.9:
            stock_df[col] = converted

stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,upcom_index_date_month_cos,upcom_index_date_dow_sin,upcom_index_date_dow_cos,upcom_index_date_hour_sin,upcom_index_date_hour_cos,upcom_index_date_quarter_sin,upcom_index_date_quarter_cos,upcom_index_date_doy_sin,upcom_index_date_doy_cos,upcom_index_date_unix_ts
0,VCB,2009-06-30,60.0,9.13,10.0,294070,17.64,60.0,60.0,60.0,...,-1.000000,0.781831,0.623490,0.0,1.0,1.224647e-16,-1.000000e+00,0.025818,-0.999667,1.246320e+09
1,VCB,2009-07-01,60.5,9.21,0.5,6248390,389.79,63.0,63.0,59.5,...,-0.866025,0.974928,-0.222521,0.0,1.0,-1.000000e+00,-1.836970e-16,0.008607,-0.999963,1.246406e+09
2,VCB,2009-07-02,58.0,8.83,-2.5,1515670,88.93,59.5,60.0,57.5,...,-0.866025,0.433884,-0.900969,0.0,1.0,-1.000000e+00,-1.836970e-16,-0.008607,-0.999963,1.246493e+09
3,VCB,2009-07-03,56.0,8.53,-2.0,899720,50.68,56.5,57.0,56.0,...,-0.866025,-0.433884,-0.900969,0.0,1.0,-1.000000e+00,-1.836970e-16,-0.025818,-0.999667,1.246579e+09
4,VCB,2009-07-06,58.5,8.91,2.5,1571740,90.18,56.0,58.5,56.0,...,-0.866025,0.000000,1.000000,0.0,1.0,-1.000000e+00,-1.836970e-16,-0.077386,-0.997001,1.246838e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4126,VCB,2026-04-22,59.4,59.40,-0.2,3011000,178.85,59.6,59.8,59.2,...,-0.500000,0.974928,-0.222521,0.0,1.0,1.224647e-16,-1.000000e+00,0.936881,-0.349647,1.776816e+09
4127,VCB,2026-04-23,62.8,62.80,3.4,35200000,2195.10,59.7,63.5,59.6,...,-0.500000,0.433884,-0.900969,0.0,1.0,1.224647e-16,-1.000000e+00,0.930724,-0.365723,1.776902e+09
4128,VCB,2026-04-24,60.6,60.60,-2.2,11882500,730.33,63.0,63.0,60.6,...,-0.500000,-0.433884,-0.900969,0.0,1.0,1.224647e-16,-1.000000e+00,0.924291,-0.381689,1.776989e+09
4129,VCB,2026-04-28,59.8,59.80,-0.8,7596400,458.90,60.8,61.5,59.8,...,-0.500000,0.781831,0.623490,0.0,1.0,1.224647e-16,-1.000000e+00,0.895839,-0.444378,1.777334e+09


## Create features

In [10]:
N_LIST = [5]

In [11]:
feature_functions = [
    lambda df: add_bbands(
        df,
        n=N_LIST,
    ),
    lambda df: add_dema(
        df,
        n=N_LIST,
    ),
    lambda df: add_ema(
        df,
        n=N_LIST,
    ),
    lambda df: add_kama(
        df,
        n=N_LIST,
    ),
    lambda df: add_midpoint(
        df,
        n=N_LIST,
    ),
    lambda df: add_midprice(
        df,
        n=N_LIST,
    ),
    lambda df: add_sar(df),
    lambda df: add_sma(
        df,
        n=N_LIST,
    ),
    lambda df: add_t3(
        df,
        n=N_LIST,
    ),
    lambda df: add_tema(
        df,
        n=N_LIST,
    ),
    lambda df: add_trima(
        df,
        n=N_LIST,
    ),
    lambda df: add_wma(
        df,
        n=N_LIST,
    ),
    lambda df: add_adx(
        df,
        n=N_LIST,
    ),
    lambda df: add_aroon(
        df,
        n=N_LIST,
    ),
    lambda df: add_bop(
        df,
        n=N_LIST,
    ),
    lambda df: add_cci(
        df,
        n=N_LIST,
    ),
    lambda df: add_cmo(
        df,
        n=N_LIST,
    ),
    lambda df: add_macd(
        df,
    ),
    lambda df: add_mfi(
        df,
        n=N_LIST,
    ),
    lambda df: add_mom(
        df,
        n=N_LIST,
    ),
    lambda df: add_ppo(
        df,
    ),
    lambda df: add_roc(
        df,
        n=N_LIST,
    ),
    lambda df: add_rsi(
        df,
        n=N_LIST,
    ),
    lambda df: add_stoch(
        df,
    ),
    lambda df: add_stoch_rsi(
        df,
        n=N_LIST,
    ),
    lambda df: add_trix(
        df,
        n=N_LIST,
    ),
    lambda df: add_ultosc(
        df,
    ),
    lambda df: add_willr(
        df,
        n=N_LIST,
    ),
    lambda df: add_ad(
        df,
        n=N_LIST,
    ),
    lambda df: add_adosc(
        df,
    ),
    lambda df: add_obv(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_dcperiod(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_dcphase(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_phasor(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_sine(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_trendmode(
        df,
        n=N_LIST,
    ),
]
len(feature_functions)

36

In [12]:
def apply_features(df, funcs):
    for func in funcs:
        df = func(df)
    return df


featured_stock_df = apply_features(stock_df, feature_functions)
featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
0,VCB,2009-06-30,60.0,9.13,10.0,294070,17.64,60.0,60.0,60.0,...,True,0.000000,NaN,0.000000,NaN,NaN,False,False,0.000000,NaN
1,VCB,2009-07-01,60.5,9.21,0.5,6248390,389.79,63.0,63.0,59.5,...,True,0.000000,0.000000,0.000000,0.000000,NaN,False,False,0.000000,0.000000
2,VCB,2009-07-02,58.0,8.83,-2.5,1515670,88.93,59.5,60.0,57.5,...,True,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
3,VCB,2009-07-03,56.0,8.53,-2.0,899720,50.68,56.5,57.0,56.0,...,True,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
4,VCB,2009-07-06,58.5,8.91,2.5,1571740,90.18,56.0,58.5,56.0,...,True,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4126,VCB,2026-04-22,59.4,59.40,-0.2,3011000,178.85,59.6,59.8,59.2,...,True,0.058528,-0.029264,-0.058528,0.029264,-0.014632,False,True,0.058528,0.000000
4127,VCB,2026-04-23,62.8,62.80,3.4,35200000,2195.10,59.7,63.5,59.6,...,True,0.372352,0.313824,0.627648,0.686176,0.656912,True,False,0.627648,0.627648
4128,VCB,2026-04-24,60.6,60.60,-2.2,11882500,730.33,63.0,63.0,60.6,...,True,0.581568,0.209216,0.418432,-0.209216,-0.895392,True,False,0.418432,0.000000
4129,VCB,2026-04-28,59.8,59.80,-0.8,7596400,458.90,60.8,61.5,59.8,...,True,0.721045,0.139477,0.278955,-0.139477,0.069739,True,False,0.278955,0.000000


In [13]:
# Drop rows with missing values
featured_stock_df = featured_stock_df.ffill().dropna()
featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
65,VCB,2009-09-30,53.5,8.14,-0.5,712000,38.11,54.0,54.5,53.0,...,True,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
66,VCB,2009-10-01,52.5,7.99,-1.0,875180,46.24,53.5,53.5,52.5,...,True,0.333333,0.333333,0.666667,0.666667,0.666667,True,False,0.666667,0.666667
67,VCB,2009-10-02,51.0,7.76,-1.5,1416100,72.10,51.5,51.5,50.0,...,True,0.555556,0.222222,0.444444,-0.222222,-0.888889,True,False,0.444444,0.000000
68,VCB,2009-10-05,51.0,7.76,0.0,606370,30.78,51.5,51.5,50.0,...,True,0.703704,0.148148,0.296296,-0.148148,0.074074,True,False,0.296296,0.000000
69,VCB,2009-10-06,50.5,7.69,-0.5,460990,23.38,51.5,51.5,50.5,...,True,0.802469,0.098765,0.197531,-0.098765,0.049383,True,False,0.197531,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4126,VCB,2026-04-22,59.4,59.40,-0.2,3011000,178.85,59.6,59.8,59.2,...,True,0.058528,-0.029264,-0.058528,0.029264,-0.014632,False,True,0.058528,0.000000
4127,VCB,2026-04-23,62.8,62.80,3.4,35200000,2195.10,59.7,63.5,59.6,...,True,0.372352,0.313824,0.627648,0.686176,0.656912,True,False,0.627648,0.627648
4128,VCB,2026-04-24,60.6,60.60,-2.2,11882500,730.33,63.0,63.0,60.6,...,True,0.581568,0.209216,0.418432,-0.209216,-0.895392,True,False,0.418432,0.000000
4129,VCB,2026-04-28,59.8,59.80,-0.8,7596400,458.90,60.8,61.5,59.8,...,True,0.721045,0.139477,0.278955,-0.139477,0.069739,True,False,0.278955,0.000000


## Split Train Val Test

In [14]:
train_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= TRAIN_RANGE[0])
    & (featured_stock_df["date"] <= TRAIN_RANGE[1])
]
train_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
65,VCB,2009-09-30,53.5,8.14,-0.5,712000,38.11,54.0,54.5,53.0,...,True,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,0.000000,0.000000
66,VCB,2009-10-01,52.5,7.99,-1.0,875180,46.24,53.5,53.5,52.5,...,True,0.333333,0.333333,0.666667,0.666667,0.666667,True,False,0.666667,0.666667
67,VCB,2009-10-02,51.0,7.76,-1.5,1416100,72.10,51.5,51.5,50.0,...,True,0.555556,0.222222,0.444444,-0.222222,-0.888889,True,False,0.444444,0.000000
68,VCB,2009-10-05,51.0,7.76,0.0,606370,30.78,51.5,51.5,50.0,...,True,0.703704,0.148148,0.296296,-0.148148,0.074074,True,False,0.296296,0.000000
69,VCB,2009-10-06,50.5,7.69,-0.5,460990,23.38,51.5,51.5,50.5,...,True,0.802469,0.098765,0.197531,-0.098765,0.049383,True,False,0.197531,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3562,VCB,2023-12-25,81.8,54.32,0.9,1301600,106.02,80.9,81.8,80.7,...,True,0.997838,0.001081,0.002162,-0.001081,0.000540,True,False,0.002162,0.000000
3563,VCB,2023-12-26,82.8,54.99,1.0,971200,80.09,82.0,82.8,82.0,...,True,0.998559,0.000721,0.001441,-0.000721,0.000360,True,False,0.001441,0.000000
3564,VCB,2023-12-27,82.7,54.92,-0.1,899600,74.62,82.9,83.2,82.5,...,True,0.999039,0.000480,0.000961,-0.000480,0.000240,True,False,0.000961,0.000000
3565,VCB,2023-12-28,82.8,54.99,0.1,703200,58.20,82.7,83.1,82.4,...,True,0.999360,0.000320,0.000640,-0.000320,0.000160,True,False,0.000640,0.000000


In [15]:
val_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= VAL_RANGE[0])
    & (featured_stock_df["date"] <= VAL_RANGE[1])
]
val_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
3567,VCB,2024-01-02,83.5,55.45,3.2,1785800,148.18,82.9,83.6,82.2,...,True,0.999715,0.000142,0.000285,-0.000142,0.000071,True,False,0.000285,0.000000
3568,VCB,2024-01-03,84.5,56.12,1.0,1373000,115.18,83.5,84.5,82.8,...,True,0.666477,-0.333238,-0.666477,-0.666762,-0.666619,False,True,0.666477,0.666477
3569,VCB,2024-01-04,85.9,57.04,1.4,2657900,227.01,84.5,86.2,84.0,...,True,0.777651,0.111174,0.222349,0.888826,1.555587,True,False,0.222349,0.222349
3570,VCB,2024-01-05,86.2,57.24,0.3,1180300,101.48,85.9,86.2,85.7,...,True,0.851768,0.074116,0.148232,-0.074116,-0.962942,True,False,0.148232,0.000000
3571,VCB,2024-01-08,86.8,57.64,0.6,1607800,139.20,86.3,86.8,86.3,...,True,0.901178,0.049411,0.098822,-0.049411,0.024705,True,False,0.098822,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3809,VCB,2024-12-25,92.4,61.36,0.3,1827700,169.71,92.2,93.8,92.2,...,True,0.421897,-0.210949,-0.421897,0.210949,0.894526,False,True,0.421897,0.000000
3810,VCB,2024-12-26,92.1,61.16,-0.3,2202900,203.22,92.5,92.8,92.0,...,True,0.281265,-0.140632,-0.281265,0.140632,-0.070316,False,True,0.281265,0.000000
3811,VCB,2024-12-27,92.2,61.23,0.1,1639900,151.65,92.5,93.0,92.2,...,True,0.187510,-0.093755,-0.187510,0.093755,-0.046877,False,True,0.187510,0.000000
3812,VCB,2024-12-30,92.0,61.10,-0.2,1706700,157.33,92.2,92.5,92.0,...,True,0.125007,-0.062503,-0.125007,0.062503,-0.031252,False,True,0.125007,0.000000


In [16]:
test_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= TEST_RANGE[0])
    & (featured_stock_df["date"] <= TEST_RANGE[1])
]
test_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
3814,VCB,2025-01-02,91.9,61.03,0.7,1630500,149.84,91.6,92.5,91.5,...,True,0.055559,-0.027779,-0.055559,0.027779,-0.013890,False,True,0.055559,0.000000
3815,VCB,2025-01-03,92.0,61.10,0.1,1402200,129.32,91.9,92.4,91.9,...,True,0.037039,-0.018520,-0.037039,0.018520,-0.009260,False,True,0.037039,0.000000
3816,VCB,2025-01-06,92.9,61.69,0.9,1936800,179.97,92.0,93.4,91.9,...,True,0.024693,-0.012346,-0.024693,0.012346,-0.006173,False,True,0.024693,0.000000
3817,VCB,2025-01-07,92.3,61.29,-0.6,1252400,116.01,93.2,93.2,92.3,...,True,0.016462,-0.008231,-0.016462,0.008231,-0.004115,False,True,0.016462,0.000000
3818,VCB,2025-01-08,92.4,61.36,0.1,1052000,96.75,92.3,92.4,91.5,...,True,0.010975,-0.005487,-0.010975,0.005487,-0.002744,False,True,0.010975,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4126,VCB,2026-04-22,59.4,59.40,-0.2,3011000,178.85,59.6,59.8,59.2,...,True,0.058528,-0.029264,-0.058528,0.029264,-0.014632,False,True,0.058528,0.000000
4127,VCB,2026-04-23,62.8,62.80,3.4,35200000,2195.10,59.7,63.5,59.6,...,True,0.372352,0.313824,0.627648,0.686176,0.656912,True,False,0.627648,0.627648
4128,VCB,2026-04-24,60.6,60.60,-2.2,11882500,730.33,63.0,63.0,60.6,...,True,0.581568,0.209216,0.418432,-0.209216,-0.895392,True,False,0.418432,0.000000
4129,VCB,2026-04-28,59.8,59.80,-0.8,7596400,458.90,60.8,61.5,59.8,...,True,0.721045,0.139477,0.278955,-0.139477,0.069739,True,False,0.278955,0.000000


## Standardization

In [17]:
ordinal_map = {"ht_dcphase_quadrant": [1, 2, 3, 4]}

In [18]:
def categorize_columns(df, ordinal_map: dict = None):
    """
    Auto-cast columns to suitable dtypes, then categorize into 3 lists.

    Parameters
    ----------
    df : pd.DataFrame
    ordinal_map : dict, optional
        {col_name: [ordered_categories]} for columns that should be ordinal.
        Example: {"size": ["S", "M", "L"], "priority": ["low", "med", "high"]}

    Returns
    -------
    numerical, nominal_categorical, ordinal_categorical : list of column names
    """
    ordinal_map = ordinal_map or {}
    df = df.copy()

    for col in df.columns:
        # --- 1. Try casting object/string columns ---
        if df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
            # Try numeric first
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().sum() / len(df) >= 0.9:  # 90%+ parseable → numeric
                df[col] = converted
            else:
                # Fall through to categorical casting below
                pass

        # --- 2. Cast to ordinal categorical ---
        if col in ordinal_map:
            df[col] = pd.Categorical(df[col], categories=ordinal_map[col], ordered=True)

        # --- 3. Cast remaining object/string → nominal categorical ---
        elif df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
            df[col] = pd.Categorical(df[col])

    # --- 4. Categorize ---
    numerical, nominal_categorical, ordinal_categorical = [], [], []

    for col in df.columns:
        dtype = df[col].dtype
        if pd.api.types.is_numeric_dtype(dtype):
            numerical.append(col)
        elif isinstance(dtype, pd.CategoricalDtype):
            if dtype.ordered:
                ordinal_categorical.append(col)
            else:
                nominal_categorical.append(col)

    return numerical, nominal_categorical, ordinal_categorical

In [19]:
numerical, nominal, ordinal = categorize_columns(train_featured_stock_df, ordinal_map)
numerical, nominal, ordinal

(['close',
  'adjust',
  'change',
  'matching_volume',
  'matching_value',
  'open',
  'high',
  'low',
  'percent_change',
  'number_of_buy_orders',
  'buy_volume',
  'average_volume_per_buy_order',
  'number_of_sell_orders',
  'sell_volume',
  'average_volume_per_sell_order',
  'net_volume',
  'date_year',
  'date_month',
  'date_day',
  'date_week',
  'date_day_of_week',
  'date_day_of_year',
  'date_quarter',
  'date_day_of_quarter',
  'date_days_to_quarter_end',
  'date_quarter_progress',
  'date_days_in_month',
  'date_days_to_month_end',
  'date_week_of_month',
  'date_days_in_year',
  'date_days_to_year_end',
  'date_year_progress',
  'date_is_leap_year',
  'date_is_month_start',
  'date_is_month_end',
  'date_is_quarter_start',
  'date_is_quarter_end',
  'date_is_year_end',
  'date_month_sin',
  'date_month_cos',
  'date_dow_sin',
  'date_dow_cos',
  'date_quarter_sin',
  'date_quarter_cos',
  'date_doy_sin',
  'date_doy_cos',
  'date_unix_ts',
  'vnindex_open',
  'vnindex_hi

In [20]:
numerical, nominal, ordinal = categorize_columns(train_featured_stock_df, ordinal_map)

numerical = [c for c in numerical if c not in ID_COLUMN and c != TARGET_COLUMN]
nominal = [c for c in nominal if c not in ID_COLUMN and c != TARGET_COLUMN]
ordinal = [c for c in ordinal if c not in ID_COLUMN and c != TARGET_COLUMN]

ordinal_categories = [ordinal_map[col] for col in ordinal]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical),
        ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal),
        ("ord", OrdinalEncoder(categories=ordinal_categories), ordinal),
    ],
    remainder="drop",
)

train_featured_scaled_stock_tensor = preprocessor.fit_transform(
    train_featured_stock_df
)  # fit+transform
val_featured_scaled_stock_tensor = preprocessor.transform(
    val_featured_stock_df
)  # transform only
test_featured_scaled_stock_tensor = preprocessor.transform(
    test_featured_stock_df
)  # transform only

In [21]:
display(train_featured_scaled_stock_tensor)
train_featured_scaled_stock_tensor.shape

array([[-0.02881417, -0.48977509, -0.29966438, ...,  1.        ,
         0.        ,  1.        ],
       [-0.06832442, -0.9496541 , -0.0925253 , ...,  1.        ,
         0.        ,  1.        ],
       [-0.1275898 , -1.40953312,  0.59411319, ...,  1.        ,
         0.        ,  2.        ],
       ...,
       [ 1.12488523, -0.12187188, -0.0615268 , ...,  1.        ,
         0.        ,  3.        ],
       [ 1.12883625,  0.06207973, -0.31083501, ...,  1.        ,
         0.        ,  0.        ],
       [ 1.03006062, -2.32929114,  0.85509979, ...,  1.        ,
         0.        ,  0.        ]], shape=(3502, 789))

(3502, 789)

In [22]:
display(val_featured_scaled_stock_tensor)
val_featured_scaled_stock_tensor.shape

array([[ 1.15649343,  2.91332961,  1.06340671, ...,  1.        ,
         0.        ,  0.        ],
       [ 1.19600368,  0.88986195,  0.53940248, ...,  1.        ,
         0.        ,  0.        ],
       [ 1.25131804,  1.25776516,  2.17044182, ...,  1.        ,
         0.        ,  0.        ],
       ...,
       [ 1.50023263,  0.06207973,  0.87820269, ...,  1.        ,
         0.        ,  3.        ],
       [ 1.49233058, -0.21384768,  0.96299795, ...,  1.        ,
         0.        ,  3.        ],
       [ 1.46072238, -0.7657025 ,  1.81006197, ...,  0.        ,
         1.        ,  3.        ]], shape=(247, 789))

(247, 789)

In [23]:
display(test_featured_scaled_stock_tensor)
test_featured_scaled_stock_tensor.shape

array([[ 1.48837956,  0.61393454,  0.86627043, ...,  1.        ,
         0.        ,  3.        ],
       [ 1.49233058,  0.06207973,  0.57646867, ...,  1.        ,
         0.        ,  0.        ],
       [ 1.52788981,  0.79788615,  1.25508462, ...,  1.        ,
         0.        ,  0.        ],
       ...,
       [ 0.25170863, -2.05336374, 13.88005776, ...,  1.        ,
         0.        ,  1.        ],
       [ 0.22010043, -0.7657025 ,  8.43932484, ...,  1.        ,
         0.        ,  1.        ],
       [ 0.22010043, -0.02989608,  6.95997761, ...,  1.        ,
         0.        ,  2.        ]], shape=(317, 789))

(317, 789)

## Roll windows

In [24]:
TOTAL_WINDOW = LOOKBACK_WINDOW + FORECAST_HORIZON
TOTAL_WINDOW

25

In [25]:
def make_windows(
    X_scaled, source_df, target_col, lookback_window, forecast_horizon, stride=1
):
    prices = source_df[target_col].values
    dates = source_df["date"].values

    total_window = lookback_window + forecast_horizon
    X_list, y_list, dates_list = [], [], []

    for i in range(0, len(X_scaled) - total_window + 1, stride):
        today_idx = i + lookback_window - 1
        future_idx = i + lookback_window + forecast_horizon - 1

        X_list.append(X_scaled[i : i + lookback_window])
        y_list.append(prices[future_idx] / prices[today_idx] - 1)
        dates_list.append(dates[today_idx])

    X = np.array(X_list)
    y = np.array(y_list)
    dates = np.array(dates_list)

    print(f"X: {X.shape} | y: {y.shape} | dates: {dates.shape}")
    return X, y, dates

In [26]:
X_train_tensor, y_train_tensor, dates_train = make_windows(
    train_featured_scaled_stock_tensor,
    train_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
    STRIDE,
)
X_val_tensor, y_val_tensor, dates_val = make_windows(
    val_featured_scaled_stock_tensor,
    val_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
    STRIDE,
)
X_test_tensor, y_test_tensor, dates_test = make_windows(
    test_featured_scaled_stock_tensor,
    test_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
    STRIDE,
)

X: (696, 20, 789) | y: (696,) | dates: (696,)
X: (45, 20, 789) | y: (45,) | dates: (45,)
X: (59, 20, 789) | y: (59,) | dates: (59,)


In [27]:
target_scaler = StandardScaler()
y_train_tensor = target_scaler.fit_transform(y_train_tensor.reshape(-1, 1)).flatten()
y_val_tensor = target_scaler.transform(y_val_tensor.reshape(-1, 1)).flatten()
y_test_tensor = target_scaler.transform(y_test_tensor.reshape(-1, 1)).flatten()

print(
    f"y_train_tensor — mean: {y_train_tensor.mean():.4f} | std: {y_train_tensor.std():.4f}"
)
print(
    f"y_val_tensor   — mean: {y_val_tensor.mean():.4f}   | std: {y_val_tensor.std():.4f}"
)
print(
    f"y_test_tensor  — mean: {y_test_tensor.mean():.4f}  | std: {y_test_tensor.std():.4f}"
)

y_train_tensor — mean: 0.0000 | std: 1.0000
y_val_tensor   — mean: -0.0736   | std: 0.4193
y_test_tensor  — mean: -0.0730  | std: 0.8556


In [28]:
type(X_train_tensor)

numpy.ndarray

In [29]:
dates_train[:1]

array(['2009-10-27T00:00:00.000000000'], dtype='datetime64[ns]')

## Validate windows

In [30]:
number_of_sample_windows = X_train_tensor.shape[0]
print(f"Number of sample windows: {number_of_sample_windows}")

Number of sample windows: 696


In [31]:
sample_idx = 0

if sample_idx < 0 or sample_idx > number_of_sample_windows - 1:
    raise ValueError(f"sample_idx must be between 0 and {number_of_sample_windows - 1}")

# ── raw index positions this sample corresponds to ──
today_idx = sample_idx + LOOKBACK_WINDOW - 1
future_idx = sample_idx + LOOKBACK_WINDOW + FORECAST_HORIZON - 1

# ── feature names output by the preprocessor ──
feature_names = (
    numerical
    + preprocessor.named_transformers_["nom"].get_feature_names_out(nominal).tolist()
    + ordinal
)

print("=" * 60)
print(f"SAMPLE INDEX: {sample_idx} / {number_of_sample_windows - 1}")
print("=" * 60)

print(f"\n── Input window dates ──")
print(f"  From : {train_featured_stock_df['date'].iloc[sample_idx]}")
print(f"  To   : {train_featured_stock_df['date'].iloc[today_idx]}  ← today")

print(f"\n── Target ──")
print(
    f"  Today  date               : {train_featured_stock_df['date'].iloc[today_idx]}"
)
print(
    f"  Future date               : {train_featured_stock_df['date'].iloc[future_idx]}"
)
print(
    f"  Today  price              : {train_featured_stock_df['adjust'].iloc[today_idx]}"
)
print(
    f"  Future price              : {train_featured_stock_df['adjust'].iloc[future_idx]}"
)
print(f"  y (standardized return)   : {y_train_tensor[sample_idx]:.6f}")

print(f"\n── X[0] — scaled input window — shape {X_train_tensor[sample_idx].shape} ──")
pd.DataFrame(
    X_train_tensor[sample_idx],
    columns=feature_names,
    index=train_featured_stock_df["date"].iloc[sample_idx : today_idx + 1].values,
)

SAMPLE INDEX: 0 / 695

── Input window dates ──
  From : 2009-09-30 00:00:00
  To   : 2009-10-27 00:00:00  ← today

── Target ──
  Today  date               : 2009-10-27 00:00:00
  Future date               : 2009-11-03 00:00:00
  Today  price              : 8.07
  Future price              : 7.84
  y (standardized return)   : -0.708451

── X[0] — scaled input window — shape (20, 789) ──


,close,change,matching_volume,matching_value,open,high,low,percent_change,number_of_buy_orders,buy_volume,...,upcom_index_date_is_month_end_false,upcom_index_date_is_month_end_true,upcom_index_date_is_quarter_start_false,upcom_index_date_is_quarter_start_true,upcom_index_date_is_quarter_end_false,upcom_index_date_is_quarter_end_true,upcom_index_date_is_year_start_false,upcom_index_date_is_year_end_false,upcom_index_date_is_year_end_true,ht_dcphase_quadrant
2009-09-30,-0.028814,-0.489775,-0.299664,-0.335150,-0.007596,-0.014163,-0.022627,-0.520268,-0.213869,-0.286852,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0
2009-10-01,-0.068324,-0.949654,-0.092525,-0.198270,-0.027373,-0.053282,-0.042641,-1.005928,-0.209587,-0.215369,...,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0
2009-10-02,-0.127590,-1.409533,0.594113,0.237120,-0.106481,-0.131519,-0.142710,-1.517422,0.229398,0.318862,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,2.0
2009-10-05,-0.127590,-0.029896,-0.433750,-0.458561,-0.106481,-0.131519,-0.142710,-0.039773,-0.214940,-0.367307,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,2.0
2009-10-06,-0.147345,-0.489775,-0.618294,-0.583150,-0.106481,-0.131519,-0.122696,-0.546101,-0.208516,-0.415357,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0
2009-10-07,-0.127590,0.429983,-0.446177,-0.463948,-0.126258,-0.131519,-0.122696,0.471720,-0.344494,-0.196741,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0
2009-10-08,-0.127590,-0.029896,-0.634022,-0.593252,-0.106481,-0.131519,-0.122696,-0.039773,-0.368049,-0.427750,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0
2009-10-09,-0.107835,0.429983,-0.194483,-0.289355,-0.106481,-0.111959,-0.102682,0.466554,-0.358413,-0.142856,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0
2009-10-12,-0.048569,1.349741,-0.239851,-0.309222,-0.086704,-0.072841,-0.082668,1.463708,-0.611097,-0.179543,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0
2009-10-13,-0.088080,-0.949654,-0.551499,-0.525738,-0.047150,-0.072841,-0.102682,-1.016261,-0.535078,-0.330151,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0


## Create metadata JSON

In [32]:
metadata = {
    "stock_code": STOCK_CODE,
    "lookback_window": LOOKBACK_WINDOW,
    "forecast_horizon": FORECAST_HORIZON,
    "stride": STRIDE,
    "train_range": [
        train_featured_stock_df.date.min().strftime("%Y-%m-%d"),
        train_featured_stock_df.date.max().strftime("%Y-%m-%d"),
    ],
    "train_shape": X_train_tensor.shape,
    "val_range": [
        val_featured_stock_df.date.min().strftime("%Y-%m-%d"),
        val_featured_stock_df.date.max().strftime("%Y-%m-%d"),
    ],
    "val_shape": X_val_tensor.shape,
    "test_range": [
        test_featured_stock_df.date.min().strftime("%Y-%m-%d"),
        test_featured_stock_df.date.max().strftime("%Y-%m-%d"),
    ],
    "test_shape": X_test_tensor.shape,
}

metadata

{'stock_code': 'vcb',
 'lookback_window': 20,
 'forecast_horizon': 5,
 'stride': 5,
 'train_range': ['2009-09-30', '2023-12-29'],
 'train_shape': (696, 20, 789),
 'val_range': ['2024-01-02', '2024-12-31'],
 'val_shape': (45, 20, 789),
 'test_range': ['2025-01-02', '2026-04-29'],
 'test_shape': (59, 20, 789)}

## Write to folder

In [33]:
TRAIN_TEST_SET_DIR

'../../train_test_set'

In [34]:
TRAIN_TEST_SET_STOCK_CODE = (
    f"{TRAIN_TEST_SET_DIR}/{STOCK_CODE}_{LOOKBACK_WINDOW}_{FORECAST_HORIZON}_{STRIDE}"
)
os.makedirs(TRAIN_TEST_SET_STOCK_CODE, exist_ok=True)
TRAIN_TEST_SET_STOCK_CODE

'../../train_test_set/vcb_20_5_5'

In [35]:
metadata_path = f"{TRAIN_TEST_SET_STOCK_CODE}/metadata.json"
print(metadata_path)
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)
    print(f"Metadata written to {metadata_path}")

../../train_test_set/vcb_20_5_5/metadata.json
Metadata written to ../../train_test_set/vcb_20_5_5/metadata.json


In [36]:
import joblib

train_featured_stock_df.to_csv(
    f"{TRAIN_TEST_SET_STOCK_CODE}/train_featured_stock_df.csv", index=False
)
val_featured_stock_df.to_csv(
    f"{TRAIN_TEST_SET_STOCK_CODE}/val_featured_stock_df.csv", index=False
)
test_featured_stock_df.to_csv(
    f"{TRAIN_TEST_SET_STOCK_CODE}/test_featured_stock_df.csv", index=False
)

np.save(
    f"{TRAIN_TEST_SET_STOCK_CODE}/train_featured_scaled_stock_tensor.npy",
    train_featured_scaled_stock_tensor,
)
np.save(
    f"{TRAIN_TEST_SET_STOCK_CODE}/val_featured_scaled_stock_tensor.npy",
    val_featured_scaled_stock_tensor,
)
np.save(
    f"{TRAIN_TEST_SET_STOCK_CODE}/test_featured_scaled_stock_tensor.npy",
    test_featured_scaled_stock_tensor,
)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_train_tensor.npy", X_train_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_val_tensor.npy", X_val_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_test_tensor.npy", X_test_tensor)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_train_tensor.npy", y_train_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_val_tensor.npy", y_val_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_test_tensor.npy", y_test_tensor)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_train.npy", dates_train)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_val.npy", dates_val)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_test.npy", dates_test)

joblib.dump(target_scaler, f"{TRAIN_TEST_SET_STOCK_CODE}/target_scaler.pkl")

['../../train_test_set/vcb_20_5_5/target_scaler.pkl']